# YouTube → Separation → Zero-Contamination Experiment

This notebook mirrors the SonicStudio **Experiment** funnel, with one processing step per cell. Run it on the model server (`vsf-242`) with the repository `.venv` kernel. Downloads, stems, previews, and saved diarization results stay under the repository `.data/` directory.

- **Flexible Model Selection:** Choose your source separation backend (`MelRoFormer`, `BSRoFormer`, `HTDemucs`, `MVSepMDX23`, or bypass separation), primary and secondary diarization engines (`Sortformer`, `DiariZen`, `Pyannote Community 1`, `Pyannote 3.1`), syllable aligner (`PhoWhisper`, `Whisper`, `MMS-FA`), and foundation verifiers (`Gemini Flash`, `Gemma 4`, `VibeVoice-ASR`).
- **Newest Zero-Contamination Config:** Includes Stage 3d Intelligent ASR & Pause-Guided Turn Segmentation (TTS sentence sizing), competitor tripwires, context-aware collars, and gate bypass safeguards.

In [ ]:
from __future__ import annotations

import os
import sys
from dataclasses import replace
from html import escape as html_escape
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Audio as IPythonAudio, HTML, clear_output, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf

# Works when Jupyter starts in src/notebooks/ as documented, and also from repo subdirectories.
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError("Could not find the repository root (pyproject.toml).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

from src.data_paths import DATA_DIR
from src.utils.AudioClass import Audio
from src.utils.AudioCutter import AudioCutter
from src.yt_crawler.YtCrawlerClass import YtCrawler

# Source Separation Backends
from src.separation import (
    BaseSeparator,
    BSRoFormer,
    HTDemucs,
    MelRoFormer,
    MVSepMDX23,
)

# Speaker Diarization Backends & Schema
from src.diarization import (
    BaseDiarizer,
    ClusteringWorkerDiarizer,
    DiariZenDiarizer,
    DiariZenWorkerDiarizer,
    DiarizationModelInfo,
    DiarizationResult,
    PyannoteDiarizer,
    SortformerDiarizer,
    SortformerWorkerDiarizer,
    Speaker,
    SpeakerTurn,
    SpeakerVerifier,
    ThreeDSpeakerWorkerDiarizer,
    ZeroContaminationConfig,
    ZeroContaminationResult,
    DEFAULT_EMBEDDING_MODEL_ID,
)

# Pipeline Stages & Defaults
from src.diarization.zero_contamination import (
    DEFAULT_COLLAR_EROSION_S,
    DEFAULT_COMPETITOR_ONSET,
    DEFAULT_ENERGY_FRAME_LEN_MS,
    DEFAULT_ENERGY_HOP_LEN_MS,
    DEFAULT_ENERGY_SEARCH_WINDOW_S,
    DEFAULT_ENERGY_VALLEY_FLOOR_DB,
    DEFAULT_HANDOFF_RISK_DISTANCE_S,
    DEFAULT_HOMOGENEITY_HOP_S,
    DEFAULT_HOMOGENEITY_WINDOW_S,
    DEFAULT_MIN_HOMOGENEITY_SIMILARITY,
    DEFAULT_MIN_SPLIT_PAUSE_S,
    DEFAULT_MIN_TURN_DURATION_S,
    DEFAULT_SILENCE_TAIL_BUFFER_S,
    DEFAULT_TARGET_MAX_DURATION_S,
    DEFAULT_TARGET_MIN_DURATION_S,
    DEFAULT_TARGET_OFFSET,
    DEFAULT_TARGET_ONSET,
    DEFAULT_TRANSITION_EXCLUSION_S,
    align_and_lock_syllable_boundaries,
    apply_context_aware_collar,
    compute_consensus_turns,
    filter_by_embedding_homogeneity,
    filter_by_foundation_models,
    run_zero_contamination_pipeline,
    smart_segment_speaker_turns,
    snap_boundaries_to_acoustic_valleys,
)


## Configuration & Global Settings

Configure audio source, device mapping, and stage gates. The values below reflect the newest `ZeroContaminationConfig` schema.

In [ ]:
# Input audio: specify YouTube URL or point to a local audio file
URL = "https://www.youtube.com/watch?v=REPLACE_ME"
LOCAL_AUDIO_PATH: Path | None = None  # e.g. Path("/path/to/local.wav")

# Device allocation per component
DEVICE = "cuda:0"
SECONDARY_DEVICE = DEVICE       # "same", "cuda:0", "cuda:1", or "cpu"
HOMOGENEITY_DEVICE = DEVICE     # "same", "cuda:0", or "cpu"
ALIGNER_DEVICE = "cpu"          # "cpu" recommended to avoid VRAM contention
VIBEVOICE_DEVICE = "cuda:1"     # dedicated secondary GPU recommended for VibeVoice-ASR

# Verifier backends & models
DIRECT_AUDIO_BACKEND = os.getenv("DIRECT_AUDIO_BACKEND", "gemini")  # "gemini" or "gemma4"
DIRECT_AUDIO_MODEL = os.getenv("DIRECT_AUDIO_MODEL") or (
    "gemini-3.8-flash" if DIRECT_AUDIO_BACKEND == "gemini" else "unsloth/gemma-4-12b-it-GGUF"
)
VIBEVOICE_MODEL_ID = "Dubedo/VibeVoice-ASR-HF-INT8"  # INT8, NF4, or BF16

# Newest ZeroContaminationConfig with all stages and thresholds exposed
config = ZeroContaminationConfig(
    device=DEVICE,
    token=os.getenv("HF_TOKEN"),
    # Stage 1: Primary Diarizer
    primary_backend="sortformer",  # "sortformer", "diarizen", "pyannote", "pyannote_31"
    primary_device=DEVICE,
    target_onset=0.80,
    target_offset=0.65,
    competitor_onset=0.20,
    # Stage 2: Dual-Engine Hungarian Consensus
    enable_consensus=True,
    secondary_backend="diarizen",  # "diarizen", "sortformer", "pyannote", "pyannote_31"
    secondary_device=SECONDARY_DEVICE,
    # Stage 3: Boundary & Collar Integrity
    enable_collar_erosion=True,
    boundary_collar_s=0.35,
    min_turn_duration_s=0.80,
    transition_exclusion_s=0.50,
    allow_gap_merge=False,
    # Stage 3a: Context-Aware Collar & Handoff Guard
    enable_context_collar=True,
    handoff_risk_distance_s=0.80,
    silence_tail_buffer_s=0.027,
    # Stage 3b: Syllable / Word Forced Alignment Lock
    enable_syllable_alignment=True,
    aligner_engine="whisper_timestamped",  # "whisper_timestamped", "mms_fa", "remote_whisper"
    aligner_model="vinai/PhoWhisper-small",  # "vinai/PhoWhisper-small", "vinai/PhoWhisper-large"
    aligner_language="vi",
    aligner_endpoint=os.getenv("WHISPER_ENDPOINT"),
    aligner_device=ALIGNER_DEVICE,
    # Stage 3c: Micro-Acoustic Energy & RMS Silence Valley Snapping
    enable_energy_snapping=True,
    energy_search_window_s=0.15,
    energy_valley_floor_db=-30.0,
    energy_frame_len_ms=2.0,
    energy_hop_len_ms=0.5,
    # Stage 3d: Intelligent ASR & Pause-Guided Turn Segmentation (TTS Sentence Sizing)
    enable_smart_segmentation=True,
    target_max_duration_s=10.0,
    target_min_duration_s=3.0,
    min_split_pause_s=0.20,
    # Stage 4: Dense Sliding-Window Embedding Homogeneity
    enable_homogeneity=True,
    homogeneity_device=HOMOGENEITY_DEVICE,
    homogeneity_window_s=1.00,
    homogeneity_hop_s=0.25,
    min_homogeneity_similarity=0.75,
    # Stage 5a: Direct-Audio Speaker Purity Verifier
    enable_gemma=True,
    gemma_backend=DIRECT_AUDIO_BACKEND,
    gemma_endpoint=os.getenv("UNSLOTH_ENDPOINT"),
    gemma_model=DIRECT_AUDIO_MODEL,
    gemma_api_key=os.getenv("GEMINI_API_KEY"),
    gemma_timeout_s=120.0,
    gemma_max_output_tokens=1024,
    # Stage 5b: VibeVoice-ASR Speaker Count Verifier
    enable_vibevoice=True,
    vibevoice_model_id=VIBEVOICE_MODEL_ID,
    vibevoice_device=VIBEVOICE_DEVICE,
    vibevoice_endpoint=os.getenv("VIBEVOICE_ENDPOINT"),
    max_secondary_speech_s=0.0,
)

stage_stats = {}
def record_stage(name: str, turns) -> list[SpeakerTurn]:
    turns = list(turns)
    stage_stats[name] = {
        "turns": len(turns),
        "speech_duration_s": round(sum(turn.duration_s for turn in turns), 2),
    }
    display(stage_stats[name])
    return turns


## Model Initializers — Flexible Backend Selection

The factories below expose explicit initialization for every separator and diarizer backend. You can either use these functions or directly instantiate your preferred model class.

In [ ]:
def init_separator(
    backend: str = "mel_roformer",
    *,
    device: str = DEVICE,
    output_dir: Path | str | None = None,
    work_dir: Path | str | None = None,
    model_name: str | None = None,
    two_stems: str = "vocals",
    **kwargs,
) -> BaseSeparator | None:
    """Initialize any supported source separator, or return None for vocal passthrough.

    Supported backends:
      - 'mel_roformer': MelRoFormer (mel_band_roformer_vocals_fv8pndlq)
      - 'bs_roformer': BSRoFormer (bs_roformer_ep_317_sdr_12.9755.ckpt)
      - 'htdemucs': HTDemucs (htdemucs_ft)
      - 'mvsep': MVSepMDX23
      - 'none' / None: Passthrough (skip separation if audio is already clean speech/vocal)
    """
    b = (backend or "").lower().strip()
    if not b or b in {"none", "passthrough"}:
        return None
    out_d = Path(output_dir) if output_dir else DATA_DIR / "notebook" / "zero_contamination" / "stems"
    wk_d = Path(work_dir) if work_dir else DATA_DIR / "notebook" / "zero_contamination" / "separation_work"

    if b in {"mel_roformer", "melroformer", "mel"}:
        return MelRoFormer(
            model_name=model_name or "mel_band_roformer_vocals_fv8pndlq",
            device=device,
            two_stems=two_stems,
            output_dir=out_d,
            work_dir=wk_d,
            **kwargs,
        )
    elif b in {"bs_roformer", "bsroformer", "bs"}:
        return BSRoFormer(
            model_name=model_name or "bs_roformer_ep_317_sdr_12.9755.ckpt",
            device=device,
            two_stems=two_stems,
            output_dir=out_d,
            work_dir=wk_d,
            **kwargs,
        )
    elif b in {"htdemucs", "demucs"}:
        return HTDemucs(
            model_name=model_name or "htdemucs_ft",
            device=device,
            two_stems=two_stems,
            output_dir=out_d,
            work_dir=wk_d,
            **kwargs,
        )
    elif b in {"mvsep", "mvsepmdx23", "mdx23"}:
        return MVSepMDX23(
            device=device,
            output_dir=out_d,
            work_dir=wk_d,
            **kwargs,
        )
    else:
        raise ValueError(
            f"Unsupported separator backend: {backend}. "
            f"Choose from 'mel_roformer', 'bs_roformer', 'htdemucs', 'mvsep', or 'none'."
        )


def init_diarizer(
    backend: str,
    *,
    device: str = "auto",
    token: str | None = None,
    onset: float = DEFAULT_TARGET_ONSET,
    offset: float = DEFAULT_TARGET_OFFSET,
    model_id: str | None = None,
    isolated: bool = True,
    **kwargs,
) -> BaseDiarizer:
    """Initialize any supported speaker diarizer backend for primary or secondary stages.

    Supported backends:
      - 'sortformer': SortformerWorkerDiarizer (isolated subprocess) or SortformerDiarizer
      - 'diarizen': DiariZenWorkerDiarizer (isolated subprocess) or DiariZenDiarizer
      - 'pyannote' / 'pyannote_community': PyannoteDiarizer (community-1)
      - 'pyannote_31': PyannoteDiarizer (v3.1)
      - 'clustering': ClusteringWorkerDiarizer
      - '3dspeaker': ThreeDSpeakerWorkerDiarizer
    """
    b = backend.lower().strip()
    tok = token or os.getenv("HF_TOKEN")
    if b in {"sortformer", "nemo-sortformer"}:
        if isolated:
            return SortformerWorkerDiarizer(device=device, token=tok, onset=onset, offset=offset, **kwargs)
        return SortformerDiarizer(device=device, token=tok, onset=onset, offset=offset, **kwargs)
    elif b in {"diarizen", "diarizen_large_s80_v2"}:
        if isolated:
            return DiariZenWorkerDiarizer(device=device, token=tok, **kwargs)
        return DiariZenDiarizer(device=device, token=tok, **kwargs)
    elif b in {"pyannote", "pyannote_community"}:
        mid = model_id or "pyannote/speaker-diarization-community-1"
        return PyannoteDiarizer(model_id=mid, device=device, token=tok, **kwargs)
    elif b in {"pyannote_31", "pyannote_3.1"}:
        mid = model_id or "pyannote/speaker-diarization-3.1"
        return PyannoteDiarizer(model_id=mid, device=device, token=tok, **kwargs)
    elif b in {"clustering", "clustering_worker"}:
        return ClusteringWorkerDiarizer(device=device, token=tok, **kwargs)
    elif b in {"3dspeaker", "3d_speaker"}:
        return ThreeDSpeakerWorkerDiarizer(device=device, token=tok, **kwargs)
    else:
        raise ValueError(
            f"Unsupported diarizer backend: {backend}. "
            f"Choose from 'sortformer', 'diarizen', 'pyannote', 'pyannote_31', 'clustering', '3dspeaker'."
        )


## Input — Load local audio or crawl YouTube URL

In [ ]:
if LOCAL_AUDIO_PATH is not None and Path(LOCAL_AUDIO_PATH).is_file():
    source_audio = Audio.from_file(LOCAL_AUDIO_PATH)
    print(f"Loaded local audio file: {source_audio.path}")
else:
    crawler = YtCrawler(
        output_dir=DATA_DIR / "notebook" / "zero_contamination" / "downloads",
        work_dir=DATA_DIR / "notebook" / "zero_contamination" / "crawl_work",
    )
    source_audio: Audio = crawler.download(URL)

display(source_audio.metadata())
source_audio.notebook_display()


## Separation — Vocals Stem Extraction

Select your separator (`mel_roformer`, `bs_roformer`, `htdemucs`, `mvsep`, or `'none'` for passthrough).

In [ ]:
# Change SEPARATION_BACKEND to 'mel_roformer', 'bs_roformer', 'htdemucs', 'mvsep', or 'none'
SEPARATION_BACKEND = "mel_roformer"

separator = init_separator(
    backend=SEPARATION_BACKEND,
    device=DEVICE,
    output_dir=DATA_DIR / "notebook" / "zero_contamination" / "stems",
    work_dir=DATA_DIR / "notebook" / "zero_contamination" / "separation_work",
)

if separator is not None:
    with separator:
        speech_audio: Audio = separator.separate(source_audio)
else:
    print("Separation skipped (passthrough mode) — using source audio directly.")
    speech_audio = source_audio

display(speech_audio.metadata())
speech_audio.notebook_display()


## Experiment Stage 1 — Primary Diarization

In [ ]:
# Initialize primary diarizer dynamically based on config.primary_backend
primary_diarizer = init_diarizer(
    backend=config.primary_backend,
    device=config.primary_device or config.device,
    token=config.token,
    onset=config.target_onset,
    offset=config.target_offset,
)

with primary_diarizer:
    primary_result: DiarizationResult = primary_diarizer.diarize(speech_audio)

current_turns = record_stage(
    "1_primary", sorted(primary_result.turns, key=lambda turn: turn.start_s)
)


## Experiment Stage 2 — Dual-Engine Mutual Hungarian Consensus

In [ ]:
if config.enable_consensus:
    sec_dev = (
        config.secondary_device
        if (config.secondary_device and config.secondary_device != "same")
        else config.device
    )
    secondary_diarizer = init_diarizer(
        backend=config.secondary_backend,
        device=sec_dev,
        token=config.token,
    )
    with secondary_diarizer:
        secondary_result: DiarizationResult = secondary_diarizer.diarize(speech_audio)
    current_turns, speaker_mapping = compute_consensus_turns(
        current_turns, secondary_result.turns, speech_audio.duration_s
    )
    current_turns = record_stage("2_consensus", current_turns)
    display({"speaker_mapping": speaker_mapping})
else:
    print("Stage 2 (Consensus) disabled in config — skipping.")


## Experiment Stage 3a — Context-Aware Collar & Handoff Guard

In [ ]:
if config.enable_context_collar:
    current_turns, collar_audits = apply_context_aware_collar(
        current_turns,
        collar_s=config.boundary_collar_s,
        handoff_risk_s=config.handoff_risk_distance_s,
        silence_tail_s=config.silence_tail_buffer_s,
        min_duration_s=config.min_turn_duration_s,
        transition_exclusion_s=config.transition_exclusion_s,
        audio_duration_s=speech_audio.duration_s,
    )
    current_turns = record_stage("3a_context_collar", current_turns)
else:
    print("Stage 3a (Context collar) disabled in config — skipping.")


## Experiment Stage 3b — Syllable / Word Forced-Alignment Lock

In [ ]:
if config.enable_syllable_alignment:
    current_turns, alignment_audits = align_and_lock_syllable_boundaries(
        speech_audio,
        current_turns,
        aligner_engine=config.aligner_engine,
        aligner_model=config.aligner_model,
        aligner_language=config.aligner_language,
        aligner_endpoint=config.aligner_endpoint,
        aligner_device=config.aligner_device or "cpu",
        token=config.token,
    )
    current_turns = record_stage("3b_word_lock", current_turns)
else:
    print("Stage 3b (Syllable alignment) disabled in config — skipping.")


## Experiment Stage 3c — Micro-Energy Valley Snapping

In [ ]:
if config.enable_energy_snapping:
    current_turns, energy_audits = snap_boundaries_to_acoustic_valleys(
        speech_audio,
        current_turns,
        search_window_s=config.energy_search_window_s,
        energy_floor_db=config.energy_valley_floor_db,
        frame_len_ms=config.energy_frame_len_ms,
        hop_len_ms=config.energy_hop_len_ms,
    )
    current_turns = record_stage("3c_energy_snap", current_turns)
else:
    print("Stage 3c (Energy snapping) disabled in config — skipping.")


## Experiment Stage 3d — Intelligent ASR & Pause-Guided Turn Segmentation (TTS Sentence Sizing)

Segments long turns into optimal TTS training slices (3–10s) using ASR punctuation and breathing pauses, snapping cut points to local acoustic energy valleys.

In [ ]:
if config.enable_smart_segmentation:
    current_turns, segment_audits = smart_segment_speaker_turns(
        speech_audio,
        current_turns,
        max_duration_s=config.target_max_duration_s,
        min_duration_s=config.target_min_duration_s,
        min_pause_s=config.min_split_pause_s,
        search_window_s=config.energy_search_window_s,
        frame_len_ms=config.energy_frame_len_ms,
        hop_len_ms=config.energy_hop_len_ms,
    )
    current_turns = record_stage("3d_smart_segmentation", current_turns)
else:
    print("Stage 3d (Smart segmentation) disabled in config — skipping.")


## Experiment Stage 4 — WeSpeaker Sliding-Window Homogeneity

In [ ]:
if config.enable_homogeneity:
    homo_dev = (
        config.homogeneity_device
        if (config.homogeneity_device and config.homogeneity_device != "same")
        else config.device
    )
    current_turns, homogeneity_audits = filter_by_embedding_homogeneity(
        speech_audio,
        current_turns,
        window_s=config.homogeneity_window_s,
        hop_s=config.homogeneity_hop_s,
        min_similarity=config.min_homogeneity_similarity,
        device=homo_dev,
        token=config.token,
    )
    current_turns = record_stage("4_homogeneity", current_turns)
else:
    print("Stage 4 (Homogeneity) disabled in config — skipping.")


## Experiment Stage 5a — Gemma/Gemini Direct-Audio Verifier

Verifies acoustic speaker purity and word completeness (không bị lẹm chữ).

In [ ]:
if config.enable_gemma:
    direct_audio_config = replace(config, enable_gemma=True, enable_vibevoice=False)
    current_turns, direct_audio_audits = filter_by_foundation_models(
        speech_audio, current_turns, direct_audio_config
    )
    current_turns = record_stage("5a_direct_audio", current_turns)
else:
    print("Stage 5a (Direct audio verifier) disabled in config — skipping.")


## Experiment Stage 5b — VibeVoice-ASR Speaker-Count Verifier

In [ ]:
if config.enable_vibevoice:
    vibevoice_config = replace(config, enable_gemma=False, enable_vibevoice=True)
    current_turns, vibevoice_audits = filter_by_foundation_models(
        speech_audio, current_turns, vibevoice_config
    )
    current_turns = record_stage("5b_vibevoice", current_turns)
else:
    print("Stage 5b (VibeVoice verifier) disabled in config — skipping.")


## Assemble and Persist Canonical Diarization Result

In [ ]:
speaker_ids = sorted({turn.speaker_id for turn in current_turns})
diarization_result = DiarizationResult(
    schema_version="2.0",
    audio_id=speech_audio.source_id,
    speakers=[Speaker(speaker_id=speaker_id) for speaker_id in speaker_ids],
    turns=current_turns,
    source_audio=speech_audio,
    model=DiarizationModelInfo(
        backend="zero-contamination-notebook",
        model_id=f"{config.primary_backend}+{config.secondary_backend}+all-experiment-gates",
    ),
)
result_path = diarization_result.save(
    DATA_DIR / "notebook" / "zero_contamination" / "results"
)
display({"saved_to": str(result_path), "funnel": stage_stats})


## Diarization result notebook viewer

The viewer brings the Experiment result table's speaker filtering, transcript search, raw/refined boundary comparison, waveform context, and lazy per-turn audio playback into Jupyter.

In [ ]:
class DiarizationResultNotebookViewer:
    """Interactive Jupyter viewer for a file-backed ``DiarizationResult``.

    Turn clips are created lazily under ``.data/notebook/diarization_viewer``.
    The controls mirror SonicStudio's result viewer: filter by speaker or
    text, inspect boundary metadata, compare blunt/refined audio, and view
    the waveform around the selected turn.
    """

    def __init__(
        self,
        result: DiarizationResult,
        output_dir: str | Path = DATA_DIR / "notebook" / "diarization_viewer",
    ) -> None:
        if result.source_audio is None:
            raise ValueError("The diarization result has no source_audio.")
        if not Path(result.source_audio.path).is_file():
            raise FileNotFoundError(result.source_audio.path)
        self.result = result
        self.audio = result.source_audio
        self.output_dir = Path(output_dir) / result.result_id
        self.cutter = AudioCutter(output_dir=self.output_dir)
        self._filtered_indices: list[int] = []

        speakers = ["All speakers", *sorted({t.speaker_id for t in result.turns})]
        self.speaker = widgets.Dropdown(options=speakers, description="Speaker:")
        self.search = widgets.Text(description="Search:", placeholder="transcript or policy")
        self.turn = widgets.Dropdown(options=[], description="Turn:", layout=widgets.Layout(width="100%"))
        self.summary = widgets.HTML()
        self.detail = widgets.Output()

        self.speaker.observe(self._filters_changed, names="value")
        self.search.observe(self._filters_changed, names="value")
        self.turn.observe(self._turn_changed, names="value")
        self._refresh_filters()

    @staticmethod
    def _transcript(turn) -> str:
        return str(getattr(turn, "_transcript", getattr(turn, "transcript", "")) or "")

    @staticmethod
    def _policy(turn) -> str:
        return str(getattr(turn, "_boundary_policy", "standard"))

    def _filters_changed(self, _change=None) -> None:
        self._refresh_filters()

    def _refresh_filters(self) -> None:
        wanted_speaker = self.speaker.value
        query = self.search.value.strip().lower()
        matches = []
        for index, turn in enumerate(self.result.turns):
            if wanted_speaker != "All speakers" and turn.speaker_id != wanted_speaker:
                continue
            haystack = f"{turn.speaker_id} {self._transcript(turn)} {self._policy(turn)}".lower()
            if query and query not in haystack:
                continue
            matches.append(index)
        self._filtered_indices = matches
        duration = sum(self.result.turns[index].duration_s for index in matches)
        average = duration / len(matches) if matches else 0.0
        self.summary.value = (
            f"<b>{len(matches)}</b> clean turns &nbsp;•&nbsp; "
            f"<b>{duration:.1f}s</b> speech &nbsp;•&nbsp; average <b>{average:.1f}s</b>"
        )
        options = [
            (f"#{index + 1} · {self.result.turns[index].speaker_id} · "
             f"{self.result.turns[index].start_s:.2f}–{self.result.turns[index].end_s:.2f}s", index)
            for index in matches
        ]
        self.turn.options = options
        if not options:
            with self.detail:
                clear_output(wait=True)
                display(HTML("<i>No turns match the active filters.</i>"))

    def _turn_changed(self, change) -> None:
        if change.get("new") is not None:
            self._render_turn(int(change["new"]))

    def _clip(self, index: int, start_s: float, end_s: float, kind: str) -> Audio:
        path = self.output_dir / f"turn_{index:06d}_{kind}.wav"
        if path.is_file():
            return Audio.from_file(path)
        return self.cutter.cut(self.audio, start_s, end_s, output_path=path)

    def _waveform(self, start_s: float, end_s: float, raw_start_s: float, raw_end_s: float) -> None:
        context_start = max(0.0, min(start_s, raw_start_s) - 0.20)
        context_end = min(self.audio.duration_s, max(end_s, raw_end_s) + 0.20)
        with sf.SoundFile(str(self.audio.path)) as source:
            sample_rate = int(source.samplerate)
            source.seek(int(context_start * sample_rate))
            waveform = source.read(int((context_end - context_start) * sample_rate), dtype="float32", always_2d=True)
        mono = waveform.mean(axis=1) if len(waveform) else np.zeros(1, dtype=np.float32)
        stride = max(1, len(mono) // 5000)
        times = context_start + np.arange(0, len(mono), stride) / sample_rate
        figure, axis = plt.subplots(figsize=(12, 2.4))
        axis.plot(times, mono[::stride], color="#64748b", linewidth=0.7)
        axis.axvspan(raw_start_s, raw_end_s, color="#f59e0b", alpha=0.18, label="Blunt/raw")
        axis.axvspan(start_s, end_s, color="#10b981", alpha=0.20, label="Refined")
        axis.set(xlabel="Time (s)", ylabel="Amplitude", xlim=(context_start, context_end))
        axis.legend(loc="upper right")
        figure.tight_layout()
        plt.show()
        plt.close(figure)

    def _render_turn(self, index: int) -> None:
        turn = self.result.turns[index]
        raw_start = float(getattr(turn, "_raw_start_s", turn.start_s))
        raw_end = float(getattr(turn, "_raw_end_s", turn.end_s))
        delta_end = float(getattr(turn, "_delta_end_ms", 0.0))
        transcript = self._transcript(turn) or "—"
        metadata = (
            "<table style='width:100%;text-align:left'>"
            f"<tr><th>Speaker</th><td>{html_escape(turn.speaker_id)}</td><th>Duration</th><td>{turn.duration_s:.2f}s</td></tr>"
            f"<tr><th>Refined</th><td>{turn.start_s:.3f}–{turn.end_s:.3f}s</td><th>Raw/blunt</th><td>{raw_start:.3f}–{raw_end:.3f}s</td></tr>"
            f"<tr><th>Policy</th><td>{html_escape(self._policy(turn))}</td><th>End delta</th><td>{delta_end:+.0f}ms</td></tr>"
            f"<tr><th>Transcript</th><td colspan='3'>{html_escape(transcript)}</td></tr></table>"
        )
        refined = self._clip(index, turn.start_s, turn.end_s, "refined")
        raw = self._clip(index, raw_start, raw_end, "raw")
        with self.detail:
            clear_output(wait=True)
            display(HTML(metadata))
            self._waveform(turn.start_s, turn.end_s, raw_start, raw_end)
            display(HTML("<b>Refined boundary</b>"))
            display(IPythonAudio(filename=str(refined.path)))
            if raw_start != turn.start_s or raw_end != turn.end_s:
                display(HTML("<b>Raw/blunt boundary</b>"))
                display(IPythonAudio(filename=str(raw.path)))

    def display(self) -> None:
        """Render the interactive viewer once and return ``None``."""
        controls = widgets.HBox([self.speaker, self.search])
        display(widgets.VBox([controls, self.summary, self.turn, self.detail]))
        if self.turn.value is not None:
            self._render_turn(int(self.turn.value))


viewer = DiarizationResultNotebookViewer(diarization_result)
viewer.display()
